In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import time

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
# ------------------------------------------------------------
# Kovasznay flow: exact steady 2D incompressible Navier-Stokes
# ------------------------------------------------------------

Re = 40.0
nu = 1.0 / Re
pi = np.pi

lam = Re / 2.0 - np.sqrt((Re**2) / 4.0 + 4.0 * pi**2)

print("Re =", Re)
print("nu =", nu)
print("lambda =", lam)

# Domain
x_min, x_max = -0.5, 1.0
y_min, y_max = -0.5, 1.5

def kovasznay_np(x, y):
    u = 1.0 - np.exp(lam * x) * np.cos(2 * pi * y)
    v = (lam / (2 * pi)) * np.exp(lam * x) * np.sin(2 * pi * y)
    p = 0.5 * (1.0 - np.exp(2 * lam * x))
    omega = ((lam**2) / (2 * pi) - 2 * pi) * np.exp(lam * x) * np.sin(2 * pi * y)
    return u, v, p, omega

def kovasznay_torch(xy):
    x = xy[:, 0:1]
    y = xy[:, 1:2]

    lam_t = torch.tensor(lam, dtype=torch.float32, device=xy.device)
    pi_t = torch.tensor(np.pi, dtype=torch.float32, device=xy.device)

    u = 1.0 - torch.exp(lam_t * x) * torch.cos(2 * pi_t * y)
    v = (lam_t / (2 * pi_t)) * torch.exp(lam_t * x) * torch.sin(2 * pi_t * y)
    p = 0.5 * (1.0 - torch.exp(2 * lam_t * x))
    omega = ((lam_t**2) / (2 * pi_t) - 2 * pi_t) * torch.exp(lam_t * x) * torch.sin(2 * pi_t * y)

    return u, v, p, omega

Re = 40.0
nu = 0.025
lambda = -0.9637405441957689


In [ ]:
def sample_interior(N):
    x = np.random.uniform(x_min, x_max, (N, 1))
    y = np.random.uniform(y_min, y_max, (N, 1))
    return torch.tensor(np.hstack([x, y]), dtype=torch.float32, device=device)

def sample_boundary(N):
    N_side = N // 4

    y_left = np.random.uniform(y_min, y_max, (N_side, 1))
    x_left = np.full_like(y_left, x_min)

    y_right = np.random.uniform(y_min, y_max, (N_side, 1))
    x_right = np.full_like(y_right, x_max)

    x_bottom = np.random.uniform(x_min, x_max, (N_side, 1))
    y_bottom = np.full_like(x_bottom, y_min)

    x_top = np.random.uniform(x_min, x_max, (N_side, 1))
    y_top = np.full_like(x_top, y_max)

    xy = np.vstack([
        np.hstack([x_left, y_left]),
        np.hstack([x_right, y_right]),
        np.hstack([x_bottom, y_bottom]),
        np.hstack([x_top, y_top])
    ])

    return torch.tensor(xy, dtype=torch.float32, device=device)

def sample_data(N):
    xy = sample_interior(N)
    with torch.no_grad():
        u, v, p, omega = kovasznay_torch(xy)
    return xy, u, v, p, omega

In [ ]:
class NSPINN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.net = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.net.append(nn.Linear(layers[i], layers[i+1]))

        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, xy):
        # scale input roughly to [-1, 1]
        x = xy[:, 0:1]
        y = xy[:, 1:2]

        x_scaled = 2.0 * (x - x_min) / (x_max - x_min) - 1.0
        y_scaled = 2.0 * (y - y_min) / (y_max - y_min) - 1.0

        z = torch.cat([x_scaled, y_scaled], dim=1)

        for layer in self.net[:-1]:
            z = torch.tanh(layer(z))

        out = self.net[-1](z)
        u = out[:, 0:1]
        v = out[:, 1:2]
        p = out[:, 2:3]
        return u, v, p

In [ ]:
def gradients(y, x):
    return torch.autograd.grad(
        y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

def ns_residuals(model, xy):
    xy = xy.clone().detach().requires_grad_(True)

    u, v, p = model(xy)

    grad_u = gradients(u, xy)
    grad_v = gradients(v, xy)
    grad_p = gradients(p, xy)

    u_x = grad_u[:, 0:1]
    u_y = grad_u[:, 1:2]
    v_x = grad_v[:, 0:1]
    v_y = grad_v[:, 1:2]
    p_x = grad_p[:, 0:1]
    p_y = grad_p[:, 1:2]

    u_xx = gradients(u_x, xy)[:, 0:1]
    u_yy = gradients(u_y, xy)[:, 1:2]
    v_xx = gradients(v_x, xy)[:, 0:1]
    v_yy = gradients(v_y, xy)[:, 1:2]

    f_u = u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
    f_v = u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)
    f_c = u_x + v_y

    omega = v_x - u_y

    return f_u, f_v, f_c, omega

def compute_loss(model, data, weights):
    xy_f, xy_bc, xy_data, u_data, v_data, p_data, omega_data = data

    # PDE residual
    f_u, f_v, f_c, omega_f = ns_residuals(model, xy_f)
    loss_pde = torch.mean(f_u**2 + f_v**2)
    loss_div = torch.mean(f_c**2)

    # Boundary loss
    u_bc_pred, v_bc_pred, p_bc_pred = model(xy_bc)
    u_bc, v_bc, p_bc, _ = kovasznay_torch(xy_bc)

    loss_bc = torch.mean((u_bc_pred - u_bc)**2 + (v_bc_pred - v_bc)**2)

    # Sparse data loss
    u_pred, v_pred, p_pred = model(xy_data)
    _, _, _, omega_pred = ns_residuals(model, xy_data)

    loss_u = torch.mean((u_pred - u_data)**2)
    loss_v = torch.mean((v_pred - v_data)**2)

    # pressure centered data loss
    p_pred_c = p_pred - torch.mean(p_pred)
    p_data_c = p_data - torch.mean(p_data)
    loss_p = torch.mean((p_pred_c - p_data_c)**2)

    loss_omega = torch.mean((omega_pred - omega_data)**2)

    total_loss = (
        weights["pde"] * loss_pde
        + weights["div"] * loss_div
        + weights["bc"] * loss_bc
        + weights["u"] * loss_u
        + weights["v"] * loss_v
        + weights["p"] * loss_p
        + weights["omega"] * loss_omega
    )

    losses = {
        "total": total_loss.item(),
        "pde": loss_pde.item(),
        "div": loss_div.item(),
        "bc": loss_bc.item(),
        "u": loss_u.item(),
        "v": loss_v.item(),
        "p": loss_p.item(),
        "omega": loss_omega.item()
    }

    return total_loss, losses

In [ ]:
def relative_l2(pred, true):
    return np.linalg.norm(pred - true) / (np.linalg.norm(true) + 1e-12)

def evaluate_model(model, nx=80, ny=80):
    x = np.linspace(x_min, x_max, nx)
    y = np.linspace(y_min, y_max, ny)
    X, Y = np.meshgrid(x, y)

    xy = np.stack([X.flatten(), Y.flatten()], axis=1)
    xy_t = torch.tensor(xy, dtype=torch.float32, device=device)

    with torch.no_grad():
        u_pred, v_pred, p_pred = model(xy_t)

    _, _, _, omega_pred = ns_residuals(model, xy_t)

    u_ref, v_ref, p_ref, omega_ref = kovasznay_np(xy[:, 0:1], xy[:, 1:2])

    u_pred = u_pred.detach().cpu().numpy()
    v_pred = v_pred.detach().cpu().numpy()
    p_pred = p_pred.detach().cpu().numpy()
    omega_pred = omega_pred.detach().cpu().numpy()

    # Pressure mean centering
    p_pred_c = p_pred - np.mean(p_pred)
    p_ref_c = p_ref - np.mean(p_ref)

    metrics = {
        "u_rel": relative_l2(u_pred, u_ref),
        "v_rel": relative_l2(v_pred, v_ref),
        "p_rel": relative_l2(p_pred_c, p_ref_c),
        "omega_rel": relative_l2(omega_pred, omega_ref),
    }

    # Residual heatmap
    f_u, f_v, f_c, _ = ns_residuals(model, xy_t)
    res = torch.sqrt(f_u**2 + f_v**2 + f_c**2).detach().cpu().numpy()

    heatmaps = {
        "X": X,
        "Y": Y,
        "u_error": np.abs(u_pred - u_ref).reshape(ny, nx),
        "v_error": np.abs(v_pred - v_ref).reshape(ny, nx),
        "p_error": np.abs(p_pred_c - p_ref_c).reshape(ny, nx),
        "omega_error": np.abs(omega_pred - omega_ref).reshape(ny, nx),
        "residual": res.reshape(ny, nx),
        "u_pred": u_pred.reshape(ny, nx),
        "v_pred": v_pred.reshape(ny, nx),
        "p_pred": p_pred_c.reshape(ny, nx),
        "omega_pred": omega_pred.reshape(ny, nx),
        "u_ref": u_ref.reshape(ny, nx),
        "v_ref": v_ref.reshape(ny, nx),
        "p_ref": p_ref_c.reshape(ny, nx),
        "omega_ref": omega_ref.reshape(ny, nx),
    }

    return metrics, heatmaps

In [ ]:
def normalize_map(A):
    return A / (np.percentile(A, 95) + 1e-12)

def controller_update(weights, metrics, heatmaps, verbose=True):
    new_weights = weights.copy()

    # Identify worst field by relative error
    field_errors = {
        "u": metrics["u_rel"],
        "v": metrics["v_rel"],
        "p": metrics["p_rel"],
        "omega": metrics["omega_rel"]
    }

    worst_field = max(field_errors, key=field_errors.get)

    # Gentle multiplicative updates
    if worst_field == "p":
        new_weights["p"] *= 1.25
        new_weights["pde"] *= 1.10

    elif worst_field == "omega":
        new_weights["omega"] *= 1.25
        new_weights["pde"] *= 1.10

    elif worst_field == "u":
        new_weights["u"] *= 1.20
        new_weights["bc"] *= 1.05

    elif worst_field == "v":
        new_weights["v"] *= 1.20
        new_weights["bc"] *= 1.05

    # If divergence/PDE residual visually remains large, maintain PDE pressure
    new_weights["div"] *= 1.02

    # Clip weights to avoid chaos, because apparently models dislike being yelled at
    for k in new_weights:
        new_weights[k] = float(np.clip(new_weights[k], 0.2, 20.0))

    if verbose:
        print("Controller action: worst field =", worst_field)
        print("Updated weights:", new_weights)

    return new_weights, worst_field

def resample_focused_points(heatmaps, N_total=12000, focus_fraction=0.4):
    N_focus = int(N_total * focus_fraction)
    N_uniform = N_total - N_focus

    # Aggregate diagnostic score
    score = (
        normalize_map(heatmaps["u_error"])
        + normalize_map(heatmaps["v_error"])
        + normalize_map(heatmaps["p_error"])
        + normalize_map(heatmaps["omega_error"])
        + normalize_map(heatmaps["residual"])
    )

    score = score.flatten()
    score = score + 1e-8
    prob = score / score.sum()

    X = heatmaps["X"].flatten()
    Y = heatmaps["Y"].flatten()

    idx = np.random.choice(len(score), size=N_focus, replace=True, p=prob)

    xy_focus = np.stack([X[idx], Y[idx]], axis=1)

    xy_uniform = np.column_stack([
        np.random.uniform(x_min, x_max, N_uniform),
        np.random.uniform(y_min, y_max, N_uniform)
    ])

    xy_new = np.vstack([xy_uniform, xy_focus])
    np.random.shuffle(xy_new)

    return torch.tensor(xy_new, dtype=torch.float32, device=device)

def resample_data_points(heatmaps, N_total=1000, focus_fraction=0.5):
    N_focus = int(N_total * focus_fraction)
    N_uniform = N_total - N_focus

    score = (
        normalize_map(heatmaps["u_error"])
        + normalize_map(heatmaps["v_error"])
        + normalize_map(heatmaps["p_error"])
        + normalize_map(heatmaps["omega_error"])
    )

    score = score.flatten() + 1e-8
    prob = score / score.sum()

    X = heatmaps["X"].flatten()
    Y = heatmaps["Y"].flatten()

    idx = np.random.choice(len(score), size=N_focus, replace=True, p=prob)
    xy_focus = np.stack([X[idx], Y[idx]], axis=1)

    xy_uniform = np.column_stack([
        np.random.uniform(x_min, x_max, N_uniform),
        np.random.uniform(y_min, y_max, N_uniform)
    ])

    xy = np.vstack([xy_uniform, xy_focus])
    np.random.shuffle(xy)

    xy_t = torch.tensor(xy, dtype=torch.float32, device=device)

    with torch.no_grad():
        u, v, p, omega = kovasznay_torch(xy_t)

    return xy_t, u, v, p, omega

In [ ]:
def train_for_epochs(model, optimizer, data, weights, epochs=1000, print_every=250):
    history = []

    for epoch in range(epochs):
        optimizer.zero_grad()

        loss, losses = compute_loss(model, data, weights)

        loss.backward()
        optimizer.step()

        history.append(losses)

        if epoch % print_every == 0:
            print(
                f"Epoch {epoch:05d} | "
                f"Total {losses['total']:.3e} | "
                f"PDE {losses['pde']:.3e} | "
                f"BC {losses['bc']:.3e} | "
                f"p {losses['p']:.3e} | "
                f"omega {losses['omega']:.3e}"
            )

    return history

In [ ]:
# ------------------------------------------------------------
# Training sizes
# ------------------------------------------------------------

N_f = 12000
N_bc = 2500
N_data = 1000

adaptive_cycles = 6
epochs_per_cycle = 1000

model = NSPINN([2, 96, 96, 96, 96, 96, 3]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

weights = {
    "pde": 1.0,
    "div": 1.0,
    "bc": 5.0,
    "u": 1.0,
    "v": 1.0,
    "p": 1.0,
    "omega": 0.5,
}

xy_f = sample_interior(N_f)
xy_bc = sample_boundary(N_bc)
xy_data, u_data, v_data, p_data, omega_data = sample_data(N_data)

metrics_log = []
weight_log = []
action_log = []

start_time = time.time()

for cycle in range(adaptive_cycles):
    print("\n" + "="*70)
    print(f"ADAPTIVE CYCLE {cycle}")
    print("="*70)

    data = (xy_f, xy_bc, xy_data, u_data, v_data, p_data, omega_data)

    train_for_epochs(
        model=model,
        optimizer=optimizer,
        data=data,
        weights=weights,
        epochs=epochs_per_cycle,
        print_every=250
    )

    metrics, heatmaps = evaluate_model(model, nx=70, ny=70)

    print("\nMetrics after cycle", cycle)
    for k, v in metrics.items():
        print(f"{k:12s}: {v:.6e}")

    metrics_record = {"cycle": cycle, **metrics}
    metrics_log.append(metrics_record)

    weights, action = controller_update(weights, metrics, heatmaps)
    action_log.append({"cycle": cycle, "action": action})
    weight_log.append({"cycle": cycle, **weights})

    # Region-aware resampling
    xy_f = resample_focused_points(heatmaps, N_total=N_f, focus_fraction=0.4)

    # Sparse observation points also refreshed toward weak regions
    xy_data, u_data, v_data, p_data, omega_data = resample_data_points(
        heatmaps,
        N_total=N_data,
        focus_fraction=0.5
    )

elapsed = time.time() - start_time
print("\nTotal training time:", elapsed / 60, "minutes")


ADAPTIVE CYCLE 0
Epoch 00000 | Total 2.364e+01 | PDE 3.141e-02 | BC 2.744e+00 | p 8.562e-02 | omega 1.685e+01
Epoch 00250 | Total 1.058e+01 | PDE 2.593e-01 | BC 4.262e-01 | p 5.552e-02 | omega 1.506e+01
Epoch 00500 | Total 2.644e-01 | PDE 4.106e-02 | BC 1.180e-02 | p 1.821e-03 | omega 2.686e-01
Epoch 00750 | Total 5.761e-02 | PDE 1.645e-02 | BC 1.712e-03 | p 2.577e-04 | omega 5.211e-02

Metrics after cycle 0
u_rel       : 9.323113e-03
v_rel       : 6.878129e-02
p_rel       : 3.099592e-02
omega_rel   : 4.960042e-02
Controller action: worst field = v
Updated weights: {'pde': 1.0, 'div': 1.02, 'bc': 5.25, 'u': 1.0, 'v': 1.2, 'p': 1.0, 'omega': 0.5}

ADAPTIVE CYCLE 1
Epoch 00000 | Total 7.480e-02 | PDE 1.813e-02 | BC 6.827e-04 | p 1.764e-04 | omega 9.741e-02
Epoch 00250 | Total 1.253e-02 | PDE 5.370e-03 | BC 3.196e-04 | p 1.140e-04 | omega 6.957e-03
Epoch 00500 | Total 1.241e-02 | PDE 4.912e-03 | BC 6.379e-04 | p 6.249e-05 | omega 4.875e-03
Epoch 00750 | Total 3.413e-03 | PDE 1.501e-03 | 